# 🛍️ Analyse Comportementale Clientèle Retail
## Notebook d'Exploration et Prototypage
**Sections : 4 → 7 | Exploration, Préparation, Modélisation, Évaluation**

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import utils
import preprocessing as pp

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('✅ Imports OK')

## 1. Chargement et Exploration (Section 4 — Features)

In [ ]:
# Charger le dataset brut
df = utils.load_data('../data/raw/customers.csv')
utils.explore_data(df)

In [ ]:
# Distribution des features numériques
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
utils.plot_distributions(df, numeric_cols[:16])

In [ ]:
# Distribution de la target (Churn)
if 'Churn' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['Churn'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
    axes[0].set_title('Distribution du Churn')
    axes[0].set_xticklabels(['Fidèle (0)', 'Churner (1)'], rotation=0)
    
    df['Churn'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
    axes[1].set_title('Proportion Churn')
    plt.tight_layout()
    plt.show()
    print(f'Taux de churn : {df["Churn"].mean():.1%}')

## 2. Qualité des Données (Section 5)

In [ ]:
# Analyse des valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct (%)': missing_pct})
missing_cols = missing_df[missing_df['count'] > 0].sort_values('pct (%)', ascending=False)

print('Colonnes avec valeurs manquantes :')
display(missing_cols)

# Visualisation
if len(missing_cols) > 0:
    missing_cols['pct (%)'].plot(kind='barh', color='steelblue')
    plt.title('% Valeurs Manquantes par Feature')
    plt.xlabel('% manquant')
    plt.tight_layout()
    plt.show()

In [ ]:
# Vérification des outliers sur SupportTickets et Satisfaction
for col in ['SupportTickets', 'Satisfaction']:
    if col in df.columns:
        print(f'\n{col} — Distribution des valeurs :')
        print(df[col].value_counts().sort_index())

## 3. Corrélation et ACP (Section 6.1)

In [ ]:
# Heatmap de corrélation
high_corr = utils.plot_correlation_heatmap(df, threshold=0.8)

In [ ]:
# ACP : réduction de dimension
from sklearn.impute import SimpleImputer

X_num = df.select_dtypes(include=[np.number]).drop(columns=['Churn', 'CustomerID'], errors='ignore')

# Imputation et scaling
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X_num)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

# ACP
X_pca, pca_model, n_comp = utils.apply_pca(X_scaled, variance_threshold=0.90)

In [ ]:
# Visualisation 2D de l'ACP
if 'Churn' in df.columns:
    utils.plot_pca_2d(X_pca, labels=df['Churn'].values, title='ACP — Projection 2D colorée par Churn')

## 4. Préparation Complète (Section 6)

In [ ]:
# Pipeline de prétraitement complet
df_clean = pp.full_preprocessing_pipeline('../data/raw/customers.csv', '../data/processed/')
print(f'Dimensions après nettoyage : {df_clean.shape}')
df_clean.head()

## 5. Modélisation (Section 7)

In [ ]:
from train_model import *

# Split train/test
X_train, X_test, y_train, y_test = split_data(df_clean, target='Churn')

In [ ]:
# Standardisation
scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

# SMOTE
X_train_res, y_train_res = handle_class_imbalance(X_train_s, y_train)

In [ ]:
# Comparaison des modèles
results = train_classifiers(X_train_res, X_test_s, y_train_res, y_test)

In [ ]:
# Matrice de confusion du meilleur modèle
best_name = max(results, key=lambda x: results[x]['auc_roc'] or 0)
y_pred_best = results[best_name]['y_pred']
utils.plot_confusion_matrix(y_test, y_pred_best, title=f'Matrice de Confusion — {best_name}')
utils.print_classification_report(y_test, y_pred_best)

In [ ]:
# Clustering K-Means
X_pca_cluster, _, _ = utils.apply_pca(X_train_s.values, variance_threshold=0.85)
best_k = find_optimal_k(X_pca_cluster)
km_model, labels = train_kmeans(X_pca_cluster, best_k)
utils.plot_pca_2d(X_pca_cluster, labels=labels, title=f'K-Means — {best_k} Clusters')

## 6. Sauvegarde et Export

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)

# Sauvegarder le meilleur modèle
best_clf = results[best_name]['model']
joblib.dump(best_clf, '../models/best_classifier.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')
joblib.dump(km_model, '../models/kmeans_model.pkl')
print('✅ Modèles sauvegardés dans models/')